In [1]:
import pandas as pd
import numpy as np
import os

data = pd.read_parquet("cleaned_data/merged.parquet")

print(data.shape)
print("\nColumns:")
print(data.columns.tolist())

data.head()

(366313, 38)

Columns:
['interval_start_utc', 'interval_end_utc', 'publish_time_utc', 'north_derated_outages_mw', 'north_forced_outages_mw', 'north_planned_outages_mw', 'north_unplanned_outages_mw', 'central_derated_outages_mw', 'central_forced_outages_mw', 'central_planned_outages_mw', 'central_unplanned_outages_mw', 'south_derated_outages_mw', 'south_forced_outages_mw', 'south_planned_outages_mw', 'south_unplanned_outages_mw', 'miso_derated_outages_mw', 'miso_forced_outages_mw', 'miso_planned_outages_mw', 'miso_unplanned_outages_mw', 'publish_time_utc_rtm', 'load_forecast', 'lmp_rtm', 'energy_rtm', 'congestion_rtm', 'loss_rtm', 'publish_time_utc_dam', 'north_wind', 'central_wind', 'south_wind', 'miso_wind', 'north_solar', 'central_solar', 'south_solar', 'miso_solar', 'lmp_dam', 'energy_dam', 'congestion_dam', 'loss_dam']


,interval_start_utc,interval_end_utc,publish_time_utc,north_derated_outages_mw,north_forced_outages_mw,north_planned_outages_mw,north_unplanned_outages_mw,central_derated_outages_mw,central_forced_outages_mw,central_planned_outages_mw,...,south_wind,miso_wind,north_solar,central_solar,south_solar,miso_solar,lmp_dam,energy_dam,congestion_dam,loss_dam
0,2023-03-25 01:55:00+00:00,2023-03-25 02:00:00+00:00,2023-03-23 05:00:00+00:00,399.0,155.0,2895.0,285.0,1999.0,3006.0,18439.0,...,0.0,9920.8,3.9,1.0,1.2,6.1,26.92,26.09,1.66,-0.83
1,2023-03-25 02:00:00+00:00,2023-03-25 02:05:00+00:00,2023-03-23 05:00:00+00:00,399.0,155.0,2895.0,285.0,1999.0,3006.0,18439.0,...,0.0,10810.9,0.0,0.0,0.0,0.0,23.70,23.31,2.24,-1.85
2,2023-03-25 02:05:00+00:00,2023-03-25 02:10:00+00:00,2023-03-23 05:00:00+00:00,399.0,155.0,2895.0,285.0,1999.0,3006.0,18439.0,...,0.0,10810.9,0.0,0.0,0.0,0.0,23.70,23.31,2.24,-1.85
3,2023-03-25 02:10:00+00:00,2023-03-25 02:15:00+00:00,2023-03-23 05:00:00+00:00,399.0,155.0,2895.0,285.0,1999.0,3006.0,18439.0,...,0.0,10810.9,0.0,0.0,0.0,0.0,23.70,23.31,2.24,-1.85
4,2023-03-25 02:15:00+00:00,2023-03-25 02:20:00+00:00,2023-03-23 05:00:00+00:00,399.0,155.0,2895.0,285.0,1999.0,3006.0,18439.0,...,0.0,10810.9,0.0,0.0,0.0,0.0,23.70,23.31,2.24,-1.85


In [2]:

cols = [
    "interval_start_utc",
    "interval_end_utc",
    "load_forecast",
    "lmp_rtm",
    "lmp_dam",
    "miso_wind",
    "miso_solar",
    "miso_derated_outages_mw",
    "miso_forced_outages_mw",
    "miso_planned_outages_mw",
    "miso_unplanned_outages_mw"
]

art_data = data[cols].copy()

art_data = art_data.sort_values("interval_start_utc").reset_index(drop=True)

print(art_data.shape)
art_data.head()

(366313, 11)


,interval_start_utc,interval_end_utc,load_forecast,lmp_rtm,lmp_dam,miso_wind,miso_solar,miso_derated_outages_mw,miso_forced_outages_mw,miso_planned_outages_mw,miso_unplanned_outages_mw
0,2023-03-25 01:55:00+00:00,2023-03-25 02:00:00+00:00,71099,28.03,26.92,9920.8,6.1,3158.0,3587.0,33776.0,3491.0
1,2023-03-25 02:00:00+00:00,2023-03-25 02:05:00+00:00,70934,25.08,23.70,10810.9,0.0,3158.0,3587.0,33776.0,3491.0
2,2023-03-25 02:05:00+00:00,2023-03-25 02:10:00+00:00,70768,27.07,23.70,10810.9,0.0,3158.0,3587.0,33776.0,3491.0
3,2023-03-25 02:10:00+00:00,2023-03-25 02:15:00+00:00,70602,26.10,23.70,10810.9,0.0,3158.0,3587.0,33776.0,3491.0
4,2023-03-25 02:15:00+00:00,2023-03-25 02:20:00+00:00,70436,24.82,23.70,10810.9,0.0,3158.0,3587.0,33776.0,3491.0


In [3]:
#check for missingness
print(art_data.isna().sum())

interval_start_utc              0
interval_end_utc                0
load_forecast                   0
lmp_rtm                      5602
lmp_dam                         0
miso_wind                       0
miso_solar                      0
miso_derated_outages_mw      5185
miso_forced_outages_mw       5185
miso_planned_outages_mw      5185
miso_unplanned_outages_mw    6049
dtype: int64


In [4]:
def generate_synthetic_market_data(
    df,
    block_size=288,   # 288 = one day of 5-minute intervals
    n_blocks=30,      # 30 blocks = about 30 days if block_size=288
    noise_scale=0.03,
    seed=42
):
    rng = np.random.default_rng(seed)

    data = df.copy().reset_index(drop=True)

    feature_cols = [c for c in data.columns if c not in ["interval_start_utc", "interval_end_utc"]]

    # possible starting points for blocks
    max_start = len(data) - block_size
    if max_start <= 0:
        raise ValueError("Data is too short for the chosen block_size.")

    block_starts = rng.integers(0, max_start, size=n_blocks)

    synthetic_blocks = []

    for start in block_starts:
        block = data.iloc[start:start + block_size].copy().reset_index(drop=True)
        synthetic_blocks.append(block)

    synthetic = pd.concat(synthetic_blocks, ignore_index=True)

    # rebuild timestamps so the synthetic data is one continuous timeline
    synthetic_start = pd.Timestamp("2027-01-01 00:00:00", tz="UTC")
    synthetic["interval_start_utc"] = pd.date_range(
        start=synthetic_start,
        periods=len(synthetic),
        freq="5min"
    )
    synthetic["interval_end_utc"] = synthetic["interval_start_utc"] + pd.Timedelta(minutes=5)

    # add light noise to numeric columns
    for col in feature_cols:
        std = synthetic[col].std()

        if pd.isna(std) or std == 0:
            continue

        noise = rng.normal(0, noise_scale * std, size=len(synthetic))
        synthetic[col] = synthetic[col] + noise

    # keep nonnegative columns from going below zero
    nonnegative_cols = [
        "load_forecast",
        "miso_wind",
        "miso_solar",
        "miso_derated_outages_mw",
        "miso_forced_outages_mw",
        "miso_planned_outages_mw",
        "miso_unplanned_outages_mw"
    ]

    for col in nonnegative_cols:
        synthetic[col] = synthetic[col].clip(lower=0)

    return synthetic

In [ ]:
synthetic_df = generate_synthetic_market_data(
    art_data,
    block_size=288,   # one day
    n_blocks=3650,      # about 10 synthetic years
    noise_scale=0.03,
    seed=42
)

print(synthetic_df.shape)
synthetic_df.head()

(1051200, 11)


,interval_start_utc,interval_end_utc,load_forecast,lmp_rtm,lmp_dam,miso_wind,miso_solar,miso_derated_outages_mw,miso_forced_outages_mw,miso_planned_outages_mw,miso_unplanned_outages_mw
0,2027-01-01 00:00:00+00:00,2027-01-01 00:05:00+00:00,69310.679782,18.458870,19.102163,4556.715153,1390.626431,8154.529592,9204.934061,7992.845916,6076.297309
1,2027-01-01 00:05:00+00:00,2027-01-01 00:10:00+00:00,69265.699030,19.625819,19.239433,4382.822851,1517.178190,8208.683691,9124.827595,7568.395841,6186.727808
2,2027-01-01 00:10:00+00:00,2027-01-01 00:15:00+00:00,68541.672200,17.883951,18.166347,4639.730056,1307.405018,8144.577681,9004.006118,7301.547617,6171.967384
3,2027-01-01 00:15:00+00:00,2027-01-01 00:20:00+00:00,69307.152655,18.223335,18.647128,4411.551830,1411.412215,8166.232713,9065.464375,7600.746017,6021.653693
4,2027-01-01 00:20:00+00:00,2027-01-01 00:25:00+00:00,70225.257470,21.202286,19.387009,4550.433090,1519.214815,8127.407510,9108.811663,7923.316176,6108.383575


In [ ]:
os.makedirs("synthetic_data", exist_ok=True)

synthetic_df.to_parquet("artificial_data/synthetic_miso.parquet", index=False)

print("Saved to artificial_data/synthetic_miso.parquet")

Saved to artificial_data/synthetic_miso.parquet
